# Expense Tracker — Exploratory Data Analysis
**Data Science Project | Synthetic Financial Data 2024**

In [ ]:
import sys
sys.path.insert(0, '../src')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as mticker
from generate_data import generate_expenses, save_dataset
from clean_data import load_data, clean, add_features, save_clean
from analyze import category_summary, monthly_trend, generate_insights

sns.set_theme(style='whitegrid')
INR = mticker.FuncFormatter(lambda x, _: f'Rs {x:,.0f}')
print('Libraries loaded OK')

## Step 1 — Generate Synthetic Data

In [ ]:
df_raw = generate_expenses(n_records=600, year=2024)
save_dataset(df_raw, '../data/raw/expenses.csv')
df_raw.head(10)

## Step 2 — Clean & Feature Engineer

In [ ]:
df = load_data('../data/raw/expenses.csv')
df = clean(df)
df = add_features(df)
save_clean(df, '../data/processed/expenses_clean.csv')
print('Shape:', df.shape)
print('Dtypes:\n', df.dtypes)
df.describe().round(2)

## Step 3 — Category Summary

In [ ]:
cat_sum = category_summary(df)
print(cat_sum)

## Step 4 — Monthly Trend Chart

In [ ]:
monthly = df.groupby('month')['amount'].sum().reset_index()
labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(monthly['month'], monthly['amount'], marker='o', lw=2.5, color='#4361ee', markersize=8)
ax.fill_between(monthly['month'], monthly['amount'], alpha=0.1, color='#4361ee')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(labels)
ax.yaxis.set_major_formatter(INR)
ax.set_title('Monthly Spending Trend', fontsize=14, fontweight='bold')
ax.set_ylabel('Amount (Rs)')
plt.tight_layout()
plt.savefig('../outputs/charts/nb_monthly_trend.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 5 — Category Pie Chart

In [ ]:
cat_totals = df.groupby('category')['amount'].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 9))
wedges, texts, autotexts = ax.pie(
    cat_totals.values, labels=cat_totals.index,
    autopct='%1.1f%%', startangle=140,
    colors=sns.color_palette('pastel', len(cat_totals)),
    wedgeprops=dict(width=0.55), pctdistance=0.75
)
for t in autotexts: t.set_fontsize(9)
ax.set_title('Spending Distribution by Category', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('../outputs/charts/nb_category_pie.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 6 — Heatmap: Category x Month

In [ ]:
pivot = df.pivot_table(values='amount', index='category', columns='month', aggfunc='sum', fill_value=0)
pivot.columns = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(pivot, ax=ax, cmap='YlOrRd', fmt='.0f',
            linewidths=0.4, linecolor='white', cbar_kws={'label': 'Amount (Rs)'})
ax.set_title('Category x Month Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/charts/nb_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 7 — Auto Insights

In [ ]:
insights = generate_insights(df)
print('\nAUTO-GENERATED INSIGHTS')
print('=' * 45)
for i, line in enumerate(insights, 1):
    print(f'{i}. {line}')

## Done!
Run `streamlit run ../app.py` for the full interactive dashboard.